<a href="https://colab.research.google.com/github/devforge82/SE3082-Lab09/blob/main/SE3082_Lab09_IT24104314.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
!nvidia-smi

Wed Oct  7 18:37:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [29]:
%%writefile hello.cu
#include <cstdio>

__global__ void hello() {
    printf("Hello from GPU thread %d\n", threadIdx.x);
}

int main(void) {
    hello<<<1, 4>>>();
    cudaDeviceSynchronize();
    return 0;
}

Overwriting hello.cu


In [30]:
!nvcc -arch=sm_75 hello.cu -o hello && ./hello

Hello from GPU thread 0
Hello from GPU thread 1
Hello from GPU thread 2
Hello from GPU thread 3


In [31]:
%%writefile ex2_add.cu
#include <cstdio>

__global__ void add(int *a, int *b, int *c) {
    *c = *a + *b;
}

int main(void) {
    int a, b, c;
    int *d_a, *d_b, *d_c;
    int size = sizeof(int);

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = 2;
    b = 7;

    cudaMemcpy(d_a, &a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, &b, size, cudaMemcpyHostToDevice);

    add<<<1, 1>>>(d_a, d_b, d_c);

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    cudaMemcpy(&c, d_c, size, cudaMemcpyDeviceToHost);

    printf("Result is %d\n", c);

    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

Overwriting ex2_add.cu


In [32]:
!nvcc -arch=sm_75 ex2_add.cu -o ex2_add && ./ex2_add

Result is 9


In [33]:
%%writefile ex3_blocks.cu
#include <cstdio>
#include <cstdlib>

#define N 512

__global__ void add(int *a, int *b, int *c) {
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    int size = N * sizeof(int);

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int*)malloc(size); random_ints(a, N);
    b = (int*)malloc(size); random_ints(b, N);
    c = (int*)malloc(size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    add<<<N, 1>>>(d_a, d_b, d_c);

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int r = 0; r < N; r++)
        printf("%d + %d = %d\n", a[r], b[r], c[r]);

    int errors = 0;
    for (int r = 0; r < N; r++)
        if (c[r] != a[r] + b[r]) errors++;

    printf("Verification: %d mismatches out of %d\n", errors, N);

    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

Overwriting ex3_blocks.cu


In [34]:
!nvcc -arch=sm_75 ex3_blocks.cu -o ex3_blocks && ./ex3_blocks

83 + 77 = 160
86 + 34 = 120
77 + 90 = 167
15 + 26 = 41
93 + 24 = 117
35 + 57 = 92
86 + 14 = 100
92 + 68 = 160
49 + 5 = 54
21 + 58 = 79
62 + 12 = 74
27 + 86 = 113
90 + 0 = 90
59 + 46 = 105
63 + 26 = 89
26 + 94 = 120
40 + 16 = 56
26 + 52 = 78
72 + 78 = 150
36 + 29 = 65
11 + 46 = 57
68 + 90 = 158
67 + 47 = 114
29 + 70 = 99
82 + 51 = 133
30 + 80 = 110
62 + 31 = 93
23 + 93 = 116
67 + 57 = 124
35 + 27 = 62
29 + 12 = 41
2 + 86 = 88
22 + 14 = 36
58 + 55 = 113
69 + 12 = 81
67 + 90 = 157
93 + 12 = 105
56 + 79 = 135
11 + 10 = 21
42 + 69 = 111
29 + 89 = 118
73 + 74 = 147
21 + 55 = 76
19 + 41 = 60
84 + 20 = 104
37 + 33 = 70
98 + 87 = 185
24 + 88 = 112
15 + 38 = 53
70 + 66 = 136
13 + 70 = 83
26 + 84 = 110
91 + 56 = 147
80 + 17 = 97
56 + 6 = 62
73 + 60 = 133
62 + 49 = 111
70 + 37 = 107
96 + 5 = 101
81 + 59 = 140
5 + 17 = 22
25 + 18 = 43
84 + 45 = 129
27 + 83 = 110
36 + 73 = 109
5 + 58 = 63
46 + 73 = 119
29 + 37 = 66
13 + 89 = 102
57 + 83 = 140
24 + 7 = 31
95 + 78 = 173
82 + 57 = 139
45 + 14 = 59
14 +

In [35]:
%%writefile ex4_threads.cu
#include <cstdio>
#include <cstdlib>

#define N 512

__global__ void addT(int *a, int *b, int *c) {
    c[threadIdx.x] = a[threadIdx.x] + b[threadIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    int size = N * sizeof(int);

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int*)malloc(size); random_ints(a, N);
    b = (int*)malloc(size); random_ints(b, N);
    c = (int*)malloc(size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    addT<<<1, N>>>(d_a, d_b, d_c);

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int r = 0; r < N; r++)
        printf("%d) %d + %d = %d\n", r, a[r], b[r], c[r]);

    int errors = 0;
    for (int r = 0; r < N; r++)
        if (c[r] != a[r] + b[r]) errors++;

    printf("Verification: %d mismatches out of %d\n", errors, N);

    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

Overwriting ex4_threads.cu


In [36]:
!nvcc -arch=sm_75 ex4_threads.cu -o ex4_threads && ./ex4_threads

0) 83 + 77 = 160
1) 86 + 34 = 120
2) 77 + 90 = 167
3) 15 + 26 = 41
4) 93 + 24 = 117
5) 35 + 57 = 92
6) 86 + 14 = 100
7) 92 + 68 = 160
8) 49 + 5 = 54
9) 21 + 58 = 79
10) 62 + 12 = 74
11) 27 + 86 = 113
12) 90 + 0 = 90
13) 59 + 46 = 105
14) 63 + 26 = 89
15) 26 + 94 = 120
16) 40 + 16 = 56
17) 26 + 52 = 78
18) 72 + 78 = 150
19) 36 + 29 = 65
20) 11 + 46 = 57
21) 68 + 90 = 158
22) 67 + 47 = 114
23) 29 + 70 = 99
24) 82 + 51 = 133
25) 30 + 80 = 110
26) 62 + 31 = 93
27) 23 + 93 = 116
28) 67 + 57 = 124
29) 35 + 27 = 62
30) 29 + 12 = 41
31) 2 + 86 = 88
32) 22 + 14 = 36
33) 58 + 55 = 113
34) 69 + 12 = 81
35) 67 + 90 = 157
36) 93 + 12 = 105
37) 56 + 79 = 135
38) 11 + 10 = 21
39) 42 + 69 = 111
40) 29 + 89 = 118
41) 73 + 74 = 147
42) 21 + 55 = 76
43) 19 + 41 = 60
44) 84 + 20 = 104
45) 37 + 33 = 70
46) 98 + 87 = 185
47) 24 + 88 = 112
48) 15 + 38 = 53
49) 70 + 66 = 136
50) 13 + 70 = 83
51) 26 + 84 = 110
52) 91 + 56 = 147
53) 80 + 17 = 97
54) 56 + 6 = 62
55) 73 + 60 = 133
56) 62 + 49 = 111
57) 70 + 37 = 

In [37]:
%%writefile ex5_vecmul.cu
#include <cstdio>
#include <cstdlib>

#define N 10000000
#define THREADS_PER_BLOCK 512

__global__ void vecMul(int *a, int *b, int *c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        c[i] = a[i] * b[i];
    }
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;

    size_t size = (size_t)N * sizeof(int);

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int*)malloc(size); random_ints(a, N);
    b = (int*)malloc(size); random_ints(b, N);
    c = (int*)malloc(size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    int blocks = (N + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK;
    printf("Launching %d blocks x %d threads\n", blocks, THREADS_PER_BLOCK);

    vecMul<<<blocks, THREADS_PER_BLOCK>>>(d_a, d_b, d_c, N);

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int i = N - 1000; i < N; i++) {
        if (i == N - 1000 || i == N - 1) {
            printf("%d) %d x %d = %d\n", i, a[i], b[i], c[i]);
        }
    }

    int mismatches = 0;
    for (int i = 0; i < N; i++) {
        if (c[i] != a[i] * b[i]) {
            mismatches++;
        }
    }
    printf("Verification: %d mismatches out of %d\n", mismatches, N);

    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

Overwriting ex5_vecmul.cu


In [38]:
!nvcc -arch=sm_75 ex5_vecmul.cu -o ex5_vecmul && ./ex5_vecmul

Launching 19532 blocks x 512 threads
9999000) 76 x 52 = 3952
9999999) 31 x 73 = 2263
Verification: 0 mismatches out of 10000000


In [39]:
%%writefile ex6_matrix.cu
#include <cstdio>
#include <cstdlib>

#define ROWS 10000
#define COLS 10000

__global__ void mulElem(int *a, int *b, int *c, int rows, int cols) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < rows && col < cols) {
        int idx = row * cols + col;
        c[idx] = a[idx] * b[idx];
    }
}

void random_ints(int *x, size_t size) {
    for (size_t i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;

    size_t count = (size_t)ROWS * COLS;
    size_t size = count * sizeof(int);

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int*)malloc(size); random_ints(a, count);
    b = (int*)malloc(size); random_ints(b, count);
    c = (int*)malloc(size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    dim3 threadsPerBlock(32, 16);
    dim3 numBlocks((COLS + threadsPerBlock.x - 1) / threadsPerBlock.x,
                   (ROWS + threadsPerBlock.y - 1) / threadsPerBlock.y);

    printf("Grid: %d x %d blocks\n", numBlocks.x, numBlocks.y);

    mulElem<<<numBlocks, threadsPerBlock>>>(d_a, d_b, d_c, ROWS, COLS);

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    size_t idx1 = (size_t)9999 * COLS + 9000;
    size_t idx2 = (size_t)9999 * COLS + 9999;
    printf("C[9999][9000] = %d x %d = %d\n", a[idx1], b[idx1], c[idx1]);
    printf("C[9999][9999] = %d x %d = %d\n", a[idx2], b[idx2], c[idx2]);

    size_t mismatches = 0;
    for (size_t i = 0; i < count; i++) {
        if (c[i] != a[i] * b[i]) {
            mismatches++;
        }
    }
    printf("Verification: %zu mismatches out of %zu\n", mismatches, count);

    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

Writing ex6_matrix.cu


In [40]:
!nvcc -arch=sm_75 ex6_matrix.cu -o ex6_matrix && ./ex6_matrix

Grid: 313 x 625 blocks
C[9999][9000] = 29 x 56 = 1624
C[9999][9999] = 86 x 21 = 1806
Verification: 0 mismatches out of 100000000
